# 내 목소리 Piper 파인튜닝 (Colab, 무료 T4)

`political-shorts` 의 `scripts/voice_build_dataset.py` 가 만든 **`dataset/`** 폴더
(`metadata.csv` + `wav/NN.wav`)를 업로드해서, 한국어 사전학습 체크포인트에서
**이어 학습(fine-tune)** 합니다. 결과물 `myvoice.onnx` + `myvoice.onnx.json` 을
내려받아 PC 의 `models/` 에 넣으면 끝.

> **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 먼저 바꾸세요.

> Piper 학습 툴체인은 버전마다 인터페이스가 바뀝니다. 이 노트북은 널리 쓰이는
> `rhasspy/piper` 클래식 레시피를 고정합니다. 실패하면 각 셀 주석의 대안
> (공식 [TRAINING.md](https://github.com/OHF-Voice/piper1-gpl/blob/main/TRAINING.md))
> 을 따르세요.

## 0. GPU 확인

In [ ]:
!nvidia-smi -L || echo 'GPU 런타임이 아닙니다 — 런타임 유형을 T4 GPU 로 바꾸세요'

## 1. Piper 학습 코드 설치
espeak-ng 가 한국어 음소를 만들어 줍니다. 설치에 3~5분.

In [ ]:
%cd /content
!apt-get -qq install -y espeak-ng >/dev/null
!git clone -q https://github.com/rhasspy/piper
%cd /content/piper/src/python
!pip -q install -e . 2>/dev/null; pip -q install piper-phonemize cython librosa 'numpy<2' 'torchmetrics<1.0' pytorch-lightning==1.9.5
!bash build_monotonic_align.sh
print('done')

## 2. 데이터셋 업로드
PC 에서 `dataset/` 폴더를 **zip** 해서(`dataset.zip`) 아래 업로드 창에 올리세요.
(우클릭 → 보내기 → 압축 폴더). 또는 Google Drive 마운트 후 경로만 바꿔도 됩니다.

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                       # dataset.zip 선택
name = next(iter(up))
os.makedirs('/content/ds', exist_ok=True)
with zipfile.ZipFile(name) as z: z.extractall('/content/ds')
# metadata.csv 위치 자동 탐색
import glob
meta = glob.glob('/content/ds/**/metadata.csv', recursive=True)[0]
DATA_DIR = os.path.dirname(meta)
print('DATA_DIR =', DATA_DIR)
print(open(meta, encoding='utf-8').read()[:300])

## 3. 전처리
`wav/NN.wav` + `metadata.csv` → 학습용 텐서. 단일 화자, 22.05kHz.

In [ ]:
%cd /content/piper/src/python
!python -m piper_train.preprocess \
  --language ko \
  --input-dir "$DATA_DIR" \
  --output-dir /content/train \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

## 4. 사전학습 체크포인트 내려받기
한국어 `ko_KR` medium 체크포인트가 있으면 그것을, 없으면 다국어/영어 medium 을
베이스로 씁니다 (보코더·프로소디 전이 효과는 그대로 있습니다).

체크포인트 목록: https://huggingface.co/datasets/rhasspy/piper-checkpoints/tree/main

In [ ]:
MODE = 'finetune'   # 'finetune' (권장, 20~40분 녹음) | 'scratch' (내 음성만, 3시간+ 필요)
BASE = None
if MODE == 'finetune':
    !pip -q install huggingface_hub
    from huggingface_hub import hf_hub_download
    REPO = 'rhasspy/piper-checkpoints'
    cands = [
        'ko/ko_KR/glados/medium/epoch=2164-step=1355080.ckpt',   # 있으면 최고
        'ko/ko_KR/kss/medium/last.ckpt',
        'en/en_US/lessac/medium/epoch=2164-step=1355080.ckpt',   # 폴백: 음색 아님, 발음/보코더 뼈대만
    ]
    for c in cands:
        try:
            BASE = hf_hub_download(REPO, c, repo_type='dataset')
            print('base =', c); break
        except Exception as e:
            print('skip', c, '->', str(e)[:80])
    assert BASE, '체크포인트를 못 받았습니다 — 위 링크에서 경로를 확인해 cands 에 넣으세요'
else:
    print('scratch 모드: 베이스 없이 학습. 녹음이 3시간 미만이면 품질이 크게 낮습니다.')


## 5. 학습
`MODE='finetune'` 이면 무료 T4로 `--batch-size 12`, 2000 epoch(1~3시간). 200 epoch
마다 체크포인트 저장 → 6번 셀로 중간 샘플을 들어보고 만족스러우면 ⏹ 로 멈추세요.
`MODE='scratch'` 는 8000 epoch로 훨씬 오래 걸리고, 녹음이 3시간+ 여야 쓸 만합니다.


In [ ]:
import os
EPOCHS = 2000 if MODE == 'finetune' else 8000
resume = f'--resume_from_checkpoint "{BASE}"' if BASE else ''
os.system(
    f'python -m piper_train --dataset-dir /content/train '
    f'--accelerator gpu --devices 1 --batch-size 12 '
    f'--validation-split 0.0 --num-test-examples 0 '
    f'--max_epochs {EPOCHS} --checkpoint-epochs 200 --precision 32 {resume}'
)


## 6. (선택) 중간 체크포인트 미리 듣기

In [ ]:
import glob
ck = sorted(glob.glob('/content/train/lightning_logs/version_*/checkpoints/*.ckpt'))[-1]
print('latest:', ck)
!python -m piper_train.infer --checkpoint "$ck" --sample-rate 22050 \
  --output-dir /content/sample <<< "오늘의 정치 뉴스를 쉽고 빠르게 정리해 드리겠습니다."
from IPython.display import Audio; Audio('/content/sample/0.wav')

## 7. ONNX 로 내보내고 다운로드

In [ ]:
import glob, shutil
ck = sorted(glob.glob('/content/train/lightning_logs/version_*/checkpoints/*.ckpt'))[-1]
!python -m piper_train.export_onnx "$ck" /content/myvoice.onnx
shutil.copy('/content/train/config.json', '/content/myvoice.onnx.json')
from google.colab import files
files.download('/content/myvoice.onnx')
files.download('/content/myvoice.onnx.json')

## 8. PC 에 설치
`myvoice.onnx` + `myvoice.onnx.json` 을 프로젝트 `models\` 에 넣고:
```ini
# .env
TTS_PROVIDER=piper
VOICE_PIPELINE=true
PIPER_BIN=tools\voice\piper\piper.exe
PIPER_MODEL=models\myvoice.onnx
```
```powershell
.\.venv\Scripts\python.exe -m political_shorts voice doctor
.\.venv\Scripts\python.exe -m political_shorts voice preview "속보입니다" --all-roles
```
